# Домашнее задание — Занятие 46

## AI-Агент в n8n: ваш собственный workflow

**LMS описание:**

В этой домашке вы создадите своего AI-агента в n8n на свободную тему — личный ассистент, бизнес-инструмент, контент-помощник или что угодно ещё. Главное — реальная польза и техническая глубина.

**Перед началом обязательно:**
1. Прочитайте файл `n8n_AI_Agent_Setup_Guide.docx` (приложен к этой домашке) — там пошаговая инструкция как я собрал свой workflow на лекции
2. Получили доступ к n8n: написали мне email → получили инвайт → прошли регистрацию
3. Прошли практику (`Practic_DL_46_lesson.ipynb`) — вы должны понимать концепции code-first vs no-code

**Что вы будете делать:**
- Придумываете тему агента (любая разумная)
- Собираете workflow в n8n с минимум 3 tools
- Хотя бы 1 tool должен быть Kazakhstan-specific (курс KZT, KZ-погода, eGov, и т.д.)
- Тестируете на 5+ запросах
- Делаете скриншоты + экспортируете JSON
- Пишете рефлексию

**Сдача:** ZIP-архив с workflow.json + скриншотами + этим заполненным notebook'ом. Дедлайн — до Занятия 47.

**Максимум баллов: 100**

---

## Часть 0 — Опишите вашего агента (10 баллов)

Перед тем как открывать n8n — продумайте на бумаге что именно вы строите. Заполните поля ниже.

### 0.1 — Тема и цель агента

Замените текст в шаблоне ниже на свой:

**Название агента:** _KZ Smart Assistant_

**Кому помогает (целевой пользователь):** _Жителям Казахстана (особенно в Астане/Алматы), которые хотят быстро получать полезную повседневную информацию — курс валют, погоду, базовые сервисы._

**Какую проблему решает:** _Пользователю приходится вручную проверять разные сайты (курс валют, погода, сервисы). Это занимает время и неудобно. Агент объединяет всё в одном интерфейсе и отвечает на естественном языке._

**Почему именно AI-агент, а не обычный workflow:** _Пользователь формулирует запросы по-разному (“сколько доллар?”, “курс usd сегодня”, “тенге к евро”) — нужен LLM для понимания намерений. Также агент решает, какие tools вызывать и может комбинировать их в одном ответе._

### 0.2 — Список выбранных tools (минимум 3)

Заполните таблицу ниже. **Минимум 3 tools, хотя бы 1 — Kazakhstan-specific.**

| # | Имя tool | Тип node | Что делает | KZ-specific? |
|---|----------|----------|------------|--------------|
| 1 | get_kzt_rate | HTTP Request | Курс валют через Нацбанк РК API | ✓ да |
| 2 | _get_weather_ | _HTTP Request_ | _Получает погоду по городу_ | _✓ да_ |
| 3 | _city_info_ | _Code_ | _Возвращает базовую инфу о городе_ | _нет_ |

_(можете добавить больше строк если делаете 4+ tools)_

**Идеи для KZ-specific tools** (если не знаете с чего начать):
- Курс тенге через [Нацбанк РК](https://nationalbank.kz/) или exchangerate.host (USD/EUR/RUB → KZT)
- Погода в KZ городах через wttr.in (Astana, Almaty, Shymkent)
- Расписание eGov-сервисов (мок ОК — главное логика)
- Поиск по [Kaspi магазину](https://kaspi.kz/shop/) (HTTP Request к публичному поиску, парсинг JSON)
- Проверка статуса посылки Kazpost (HTTP Request)
- Расписание автобусов/метро Астаны (мок ОК)

## Часть 1 — System Message агента (15 баллов)

System Message — это инструкция вашему AI Agent. От её качества зависит 70% успеха.

**Требования:**
- Опишите роль агента (кто он)
- Опишите когда какие tools использовать
- Укажите язык ответа (важно для русско/казахоязычных пользователей)
- Скажите что делать когда агент НЕ знает ответ (не галлюцинировать)

Вставьте сюда вашу финальную версию System Message:

```
[You are KZ Smart Assistant — an AI assistant that helps users in Kazakhstan with daily information such as currency exchange rates, weather, and city-related info.

Your responsibilities:
- Answer user questions clearly and concisely
- Use available tools whenever real-time or external data is needed
- Combine multiple tools if necessary

Tools usage rules:
- For ANY currency-related questions (USD, EUR, RUB to KZT), ALWAYS use the get_kzt_rate tool. Never rely on your internal knowledge for rates.
- For weather-related queries, ALWAYS use the get_weather tool.
- Use city_info tool for general information about cities if needed.

Language:
- Always respond in the same language the user used (Russian or English).

Accuracy:
- If tool data is missing or unclear, say you don’t have enough information.
- Never hallucinate numbers, rates, or weather.

Behavior:
- Be helpful, short, and practical
- If a question requires multiple steps — think and call tools sequentially]
```

**Подсказка:** хороший SM обычно 100-300 слов. Короче — не хватает контекста, длиннее — модель теряется.

**Примеры паттернов которые можно использовать в SM:**
- *"Always respond in the same language the user wrote to you."*
- *"If you don't have enough information from tools — say so honestly, don't make up data."*
- *"For currency-related questions, always use the get_kzt_rate tool — never guess rates from your training data."*

## Часть 2 — Tool Descriptions (15 баллов)

По description модель решает когда вызвать tool. Плохой description = агент не вызывает tool / вызывает не вовремя.

Для каждого из ваших 3+ tools заполните блок ниже:

### Tool 1 — _get_kzt_rate_

**Тип node:** _HTTP Request_

**Description (что вы написали в поле Description tool):**
```
[Get real-time exchange rate for a given currency (USD, EUR, RUB) to Kazakhstani Tenge (KZT). Use this tool whenever user asks about currency rates.]
```

**Параметры через $fromAI() (если использовали):**
```
[{{ $fromAI('currency', 'Currency code like USD, EUR, RUB', 'string') }}]
```

---

### Tool 2 — _get_weather_

**Тип node:** _HTTP Request_

**Description:**
```
[Get current weather for a given city in Kazakhstan. Use for any weather-related questions.]
```

**Параметры через $fromAI():**
```
{{ $fromAI('city', 'City name in English (Astana, Almaty, Shymkent)', 'string') }}
```

---

### Tool 3 — _city_info_

**Тип node:** _Code_

**Description:**
```
[Provides general static information about major cities in Kazakhstan such as population and status (capital, etc.).]
```

**Параметры через $fromAI():**
```
{{ $fromAI('city', 'City name', 'string') }}
```

## Часть 3 — Тестовые запросы (15 баллов)

Прогоните агента на минимум 5 разных запросах. Запросы должны проверять разные сценарии:

1. **Простой запрос** — агент использует один tool
2. **Композиция** — агент должен использовать 2+ tools в одном запросе
3. **Memory тест** — задаёте follow-up вопрос на основе предыдущего ответа
4. **Edge case** — что-то нестандартное (агент должен обработать или честно сказать что не может)
5. **Свободный** — на ваше усмотрение

Для каждого запроса заполните таблицу:

| # | Запрос пользователя | Какие tools агент вызвал | Финальный ответ | Корректно? |
|---|---------------------|--------------------------|-----------------|------------|
| 1 | Какой сейчас курс доллара? | get_kzt_rate('USD') | 1 USD = 478.50 KZT | ✓ |
| 2 | _ваш запрос_ | _что вызвал_ | _ответ_ | _да/нет_ |
| 3 | _ваш запрос_ | _что вызвал_ | _ответ_ | _да/нет_ |
| 4 | _ваш запрос_ | _что вызвал_ | _ответ_ | _да/нет_ |
| 5 | _ваш запрос_ | _что вызвал_ | _ответ_ | _да/нет_ |

**Если агент сбойнул на каком-то запросе — это нормально и даже хорошо.** Главное чтобы вы это зафиксировали и в Части 6 описали как пытались починить.

## Часть 4 — Скриншоты (10 баллов)

Сделайте 2 обязательных скриншота и положите в ZIP вместе с workflow.json:

### Скриншот 1 — `01_workflow_canvas.png`

Полный вид вашего workflow на canvas. Должны быть видны:
- Chat Trigger node
- AI Agent node
- Chat Model (OpenAI / Gemini / другой)
- Memory (если использовали)
- Все ваши tools (минимум 3)
- Линии соединения между nodes

*Совет: используйте zoom-out (-) чтобы всё влезло в кадр.*

### Скриншот 2 — `02_execution_log.png`

Execution log одного из тестовых запросов. Откройте AI Agent node после запуска → Logs tab → раскройте дерево вызовов tools. Должно быть видно:
- Какой запрос был отправлен агенту
- Какие tools агент вызвал
- Какие аргументы агент передал в tools (через $fromAI)
- Финальный текст ответа

### Бонусные скриншоты (опционально):
- `03_tool_config.png` — настройки одного из ваших tools (Description + параметры)
- `04_chat_demo.png` — скриншот диалога с агентом через встроенный чат n8n

**В ZIP положите все скриншоты в папку `screenshots/`.**

## Часть 5 — Экспорт workflow (10 баллов)

Я должен иметь возможность импортировать ваш workflow в свой n8n и проверить.

### Как экспортировать:

1. Откройте ваш workflow в n8n
2. Кликните на имя workflow вверху → откроется dropdown
3. Выберите **Download** → файл скачается как `My Agent.json`
4. Переименуйте в `workflow.json`
5. Положите в корень ZIP-архива

### Что проверить перед сдачей:

- Файл валидный JSON (откройте в текстовом редакторе — должно начинаться с `{` и заканчиваться `}`)
- В JSON НЕ должно быть ваших API ключей (n8n экспортирует только references на credentials, сами ключи остаются на сервере — это нормально)
- Имена nodes понятные (не `HTTP Request1`, `HTTP Request2` — а `get_kzt_rate`, `get_weather`)

**Сейчас прямо тут запишите хеш или версию вашего экспорта чтобы я знал какую версию проверять:**

**Дата экспорта:** _(01.05.2026)_

**Краткое описание состояния workflow:** _(например: 'все 3 tools работают, прошёл 5 тестов')_

## Часть 6 — Рефлексия (15 баллов)

Напишите 200-300 слов о вашем опыте сборки агента. Минимум 4 пункта ниже должны быть раскрыты:

**1. Что было сложнее всего?**
_(конкретные технические или концептуальные сложности — настройка $fromAI, OAuth, agent зацикливался, и т.д.)_

**2. Что вас неожиданно удивило?**
_(что-то что вы думали будет легко, а оказалось сложно — или наоборот)_

**3. Что бы вы добавили если бы было больше времени?**
_(идеи для расширения — больше tools, sub-agents, человек-в-цикле, и т.д.)_

**4. Сравнение с code-first (Colab + LangChain) из практики:**
_(где n8n был быстрее, где LangChain был бы лучше, какой подход выбрали бы для своего проекта)_

**5. Где вы реально могли бы использовать этого агента?**
_(не теоретически — конкретный сценарий: ваша работа, учёба, личная задача)_

Ваш ответ:

```
[Самым сложным этапом при сборке агента в n8n для меня стала настройка корректной работы tools, особенно использование выражения $fromAI(). Сначала агент либо не передавал параметры в HTTP Request, либо подставлял их неправильно (например, не тот формат города или валюты). Также возникала проблема, когда агент не вызывал tool, а пытался ответить сам — пришлось дорабатывать System Message и description tools.

Неожиданно удивило, насколько сильно поведение агента зависит от описания (description) tools и System Message. Я думала, что это второстепенные поля, но оказалось, что без чётких инструкций агент работает нестабильно. С другой стороны, было приятно, что базовый рабочий прототип можно собрать довольно быстро без написания кода.

Если бы у меня было больше времени, я бы добавила дополнительные tools — например, получение курсов криптовалют, интеграцию с API Kaspi или eGov. Также было бы интересно реализовать более продвинутую memory (например, через базу данных) и добавить multi-step сценарии с несколькими агентами.

По сравнению с code-first подходом (Colab + LangChain), n8n оказался значительно быстрее и проще для прототипирования благодаря визуальному интерфейсу. Однако LangChain даёт больше гибкости и контроля, особенно для сложной логики, кастомных агентов и продакшн-решений. Для быстрого MVP я бы выбрала n8n, а для масштабного проекта — code-first подход.

Я могла бы использовать такого агента в повседневной жизни — например, для быстрого получения курса валют и погоды в одном интерфейсе, без необходимости заходить на разные сайты или приложения.]
```

## Часть 7 — Теоретические вопросы (10 баллов)

Ответы должны быть короткими (1-3 предложения). Привязаны к материалу лекции и гайду.

### Вопрос 1 (2 балла)
**В каких ситуациях code-first (LangChain) выигрывает у no-code (n8n)?** Назовите 2 конкретных случая.

Ответ:

Code-first (LangChain) выигрывает, когда нужна сложная логика (кастомные цепочки, агенты с условиями, циклы) и при интеграции с нестандартными API или ML-моделями. Также он лучше подходит для продакшена, где требуется контроль, масштабируемость и версияция кода.

---

### Вопрос 2 (2 балла)
**Что делает выражение `$fromAI('city', 'City name', 'string')` в HTTP Request tool?** Объясните своими словами.

Ответ:

$fromAI(...) позволяет агенту автоматически извлекать параметр из запроса пользователя и передавать его в tool. В данном случае агент сам определяет название города из текста и подставляет его в HTTP-запрос.

---

### Вопрос 3 (2 балла)
**Почему параметр `Max Iterations` в AI Agent node важен?** Что произойдёт если поставить 100?

Ответ:

Max Iterations ограничивает количество шагов (вызовов tools), которые может сделать агент. Если поставить 100, агент может зациклиться, дольше отвечать и тратить больше ресурсов (или токенов).

---

### Вопрос 4 (2 балла)
**Какая разница между Window Buffer Memory и Postgres Chat Memory?** Когда какой использовать?

Ответ:

Window Buffer Memory хранит только последние сообщения в рамках одной сессии (временная память). Postgres Chat Memory сохраняет историю в базе данных и подходит для долгосрочного хранения и работы с пользователем между сессиями.

---

### Вопрос 5 (2 балла)
**Почему description tool критичен для качества агента?** Приведите пример хорошего и плохого description.

Ответ:

Description определяет, когда агент будет вызывать tool — если он плохой, агент может не использовать tool или использовать его неправильно.
Хороший пример: "Get real-time weather for a city. Use for any weather-related queries."
Плохой пример: "Weather tool"

## Чек-лист перед сдачей

Перед отправкой ZIP проверьте:

- [ ] В ZIP лежит файл `Homework_46_lesson.ipynb` (этот) с заполненными ответами
- [ ] В корне ZIP лежит `workflow.json` (экспорт из n8n)
- [ ] В папке `screenshots/` минимум 2 скриншота: `01_workflow_canvas.png` и `02_execution_log.png`
- [ ] Имя ZIP: `HW46_<your_lastname>_<your_firstname>.zip` (например `HW46_Akhmetov_Olzhas.zip`)
- [ ] Все 7 частей этой домашки заполнены
- [ ] Workflow в n8n активирован (Active toggle = ON) на момент сдачи

## Критерии оценки (итого 100 баллов)

| Часть | Что оценивается | Баллов |
|-------|-----------------|--------|
| 0 | Описание агента и список tools | 10 |
| 1 | Качество System Message | 15 |
| 2 | Качество Tool Descriptions (по 5 за каждый из 3) | 15 |
| 3 | 5 разных тестовых запросов | 15 |
| 4 | Скриншоты (2 обязательных) | 10 |
| 5 | Корректный экспорт workflow.json | 10 |
| 6 | Глубина рефлексии | 15 |
| 7 | Теоретические вопросы (5 × 2) | 10 |

**Бонусные баллы:**
- +5 баллов если использовали 4+ tools
- +5 баллов если есть tool с OAuth (Google/Slack/Notion/Telegram)
- +5 баллов если есть композиция (один tool вызывает другой через workflow link)
- +5 баллов если сделали бонусные скриншоты

**Максимум бонусов: 20.** Итого с бонусами можно набрать до 120 (но в LMS зафиксируется как 100%).

---

## Удачи!

Если что-то не получается — пишите в чат курса, разберёмся вместе. Помните: ошибки агента это нормально, **главное — научиться их диагностировать**. Это и есть skill, который ценится в индустрии.

Дедлайн: до начала Занятия 47.